<a href="https://colab.research.google.com/github/eli576/USFQ_Python/blob/main/Taller_CC_Deber_06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install deap
import deap
from deap import base, creator, tools
import random


In [ ]:
# Problem set up


# Aulas: id: nombre, capacidad
aulas = {
    1: ("A", 15),
    2: ("B", 30),
    3: ("C", 20),
    4: ("D", 25),
}

# Profesores: id: nombre
profes = {
    1: "Prof. García",
    2: "Prof. Martínez",
    3: "Prof. López",
    4: "Prof. Ramos",
}

# Franjas horarias: id: día, hora
horarios = {
    1: ("Lunes", "9:00–11:00"),
    2: ("Lunes", "11:00–13:00"),
    3: ("Lunes", "13:00–15:00"),
    4: ("Martes", "9:00–11:00"),
    5: ("Martes", "11:00–13:00"),
    6: ("Martes", "13:00–15:00"),
    7: ("Miércoles", "9:00–11:00"),
    8: ("Miércoles", "11:00–13:00"),
    9: ("Miércoles", "13:00–15:00"),
    10: ("Jueves", "9:00–11:00"),
    11: ("Jueves", "11:00–13:00"),
    12: ("Jueves", "13:00–15:00"),
    13: ("Viernes", "9:00–11:00"),
    14: ("Viernes", "11:00–13:00"),
    15: ("Viernes", "13:00–15:00"),
}

# Asignaturas: id: abreviacion, nombre, profes
asignaturas = {
    1: ("CS1",  "Introducción a la Computación", [1, 2]),
    2: ("IN1",  "Inglés Introductorio",          [1, 3]),
    3: ("MA1",  "Geometría",                     [1, 2]),
    4: ("FIS1", "Física Introductoria",          [3, 4]),
    5: ("HIA",  "Historia del Arte",             [4]),
    6: ("DRA",  "Drama",                         [1, 4]),
}

# Grupos: id: num estudiantes, asignaturas
grupos = {
    1: (10, [1, 3, 4]),
    2: (30, [2, 3, 5, 6]),
    3: (18, [3, 4, 5]),
    4: (25, [1, 4]),
    5: (20, [2, 3, 5]),
    6: (22, [1, 4, 5]),
    7: (16, [1, 3]),
    8: (18, [2, 6]),
    9: (24, [1, 6]),
    10:(25, [3, 4]),
}

In [ ]:

"""
1. Como cada grupo atiende a sus asignaturas en conjunto, sabemos que el número de clases en total deben ser 26.
2. Podemos usar tres genes para cada clase:
  a. 1 gen para el id del aula
  b. 1 gen para el id de la franja horaria
  c. 1 gen para el id del profesor.
"""

# Clases: lista (grupo, asignatura)
# la lista representa cada clase que hay que alocar a un horario


clases = []  # [(grupo, asignatura), ...]
for grupo, (n_estud, asignaturas) in grupos.items():
    for asignatura in asignaturas:
        clases.append((grupo, asignatura))

# debería ser 26
n_clases = len(clases)
# 3 genes por clase: aula, horario, profesor
n_genes = n_clases * 3

In [ ]:
# Genetic algorithm set up

# minimizar
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()

# Atributos: aula, horario, profesor
toolbox.register("attr_aula",  random.randint, 1, len(aulas))
toolbox.register("attr_horario",  random.randint, 1, len(horarios))
toolbox.register("attr_profe",  random.randint, 1, len(profes))

def random_class_assignment():
    # configuracion de una clase random
    return [toolbox.attr_aula(), toolbox.attr_horario(), toolbox.attr_profe()]

def complete_individual():
    # creacion de un individuo (horario) completo - 26 clases, cada una con 3 genes
    genes = []
    for clase in range(n_clases):
        genes.extend(random_class_assignment())
    return creator.Individual(genes)

toolbox.register("individual", complete_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

In [ ]:
# Fitness function

# Fitness = número de violaciones (menor es mejor) -> minimizar

"""
Restricciones duras:
1. Un profesor no puede impartir dos clases en la misma franja horaria.
2. Un aula no puede albergar más de una clase simultáneamente.
3. Cada curso se reúne una vez por semana.
4. Cada grupo asiste a sus respectivas asignaturas en conjunto.
5. La capacidad del aula debe ser mayor o igual al tamaño del grupo asignado.
6. No usar base de datos relacional ni de ninguna otra índole.
7. Usar archivos es permisible.
"""

def evaluate(individual):
    # Calcula cuántas restricciones se violan.

    violaciones = 0

    # Diccionarios para detectar colisiones
    prof_time = {}   # (prof, slot) -> count
    room_time = {}   # (room, slot) -> count
    group_time = {}  # (group, slot) -> count

    # Recorremos cada clase
    for i, (group_id, subj_id) in enumerate(CLASS_DEMANDS):
        base_idx = i * 3
        room_id  = individual[base_idx]
        slot_id  = individual[base_idx + 1]
        prof_id  = individual[base_idx + 2]

        # Validar que los ids estén dentro de rango (por si mutación los saca)
        if room_id not in ROOMS:
            violations += 1
            continue
        if slot_id not in TIMESLOTS:
            violations += 1
            continue
        if prof_id not in PROFES:
            violations += 1
            continue

        # 1) Capacidad del aula
        n_students = GROUPS[group_id][0]
        room_capacity = ROOMS[room_id][1]
        if n_students > room_capacity:
            violations += 1

        # 2) Profesor válido para asignatura
        valid_profs = SUBJECTS[subj_id][2]
        if prof_id not in valid_profs:
            violations += 1

        # 3) Colisiones profesor-tiempo
        key_pt = (prof_id, slot_id)
        prof_time[key_pt] = prof_time.get(key_pt, 0) + 1

        # 4) Colisiones aula-tiempo
        key_rt = (room_id, slot_id)
        room_time[key_rt] = room_time.get(key_rt, 0) + 1

        # 5) Colisiones grupo-tiempo
        key_gt = (group_id, slot_id)
        group_time[key_gt] = group_time.get(key_gt, 0) + 1

    # Contar colisiones (si hay 2 clases con la misma clave -> 1 violación, etc.)
    for count in prof_time.values():
        if count > 1:
            violations += (count - 1)

    for count in room_time.values():
        if count > 1:
            violations += (count - 1)

    for count in group_time.values():
        if count > 1:
            violations += (count - 1)

    # Aquí se podrían añadir restricciones blandas como penalizaciones extra.

    return (violations,)

toolbox.register("evaluate", evaluate)

In [3]:


# =========================
# OPERADORES GENÉTICOS
# =========================

toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("mate", tools.cxTwoPoint)

def mutate(individual, indpb=0.1):
    """
    Mutación: con probabilidad indpb cambia cada gen
    (aula, franja o profesor) por un valor aleatorio válido.
    """
    for i in range(len(individual)):
        if random.random() < indpb:
            gene_pos = i % 3  # 0 = aula, 1 = franja, 2 = profesor
            if gene_pos == 0:        # aula
                individual[i] = random.randint(1, len(ROOMS))
            elif gene_pos == 1:      # franja horaria
                individual[i] = random.randint(1, len(TIMESLOTS))
            else:                    # profesor
                individual[i] = random.randint(1, len(PROFES))
    return (individual,)

toolbox.register("mutate", mutate)

# =========================
# FUNCIÓN PARA IMPRIMIR HORARIO
# =========================

def print_schedule(individual):
    print("\n================ HORARIO GENERADO ================\n")
    for i, (group_id, subj_id) in enumerate(CLASS_DEMANDS):
        base_idx = i * 3
        room_id  = individual[base_idx]
        slot_id  = individual[base_idx + 1]
        prof_id  = individual[base_idx + 2]

        # Validaciones básicas por si algo queda fuera de rango
        if room_id not in ROOMS or slot_id not in TIMESLOTS or prof_id not in PROFES:
            print(f"Clase {i}: configuración inválida (ids fuera de rango)")
            continue

        subj_abbr, subj_name, _ = SUBJECTS[subj_id]
        room_name, _ = ROOMS[room_id]
        day, hour = TIMESLOTS[slot_id]
        prof_name = PROFES[prof_id]

        print(f"Asig: {subj_abbr:3} | Grupo: {group_id:2} | "
              f"Aula: {room_name:6} | Prof: {prof_name:12} | "
              f"{day:10} {hour}")

    print("\n==================================================\n")

# =========================
# GA PRINCIPAL
# =========================

def main():
    random.seed(42)

    POP_SIZE = 150    # tamaño de población
    NGEN     = 200    # número de generaciones
    CXPB     = 0.7    # probabilidad de cruza
    MUTPB    = 0.3    # probabilidad de mutación de un individuo

    pop = toolbox.population(n=POP_SIZE)

    # Evaluar población inicial
    for ind in pop:
        ind.fitness.values = toolbox.evaluate(ind)

    print("Población inicial evaluada")

    for gen in range(1, NGEN + 1):
        # Selección
        offspring = toolbox.select(pop, len(pop))
        offspring = list(map(toolbox.clone, offspring))

        # Cruza
        for c1, c2 in zip(offspring[0::2], offspring[1::2]):
            if random.random() < CXPB:
                toolbox.mate(c1, c2)
                del c1.fitness.values
                del c2.fitness.values

        # Mutación
        for mut in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(mut)
                del mut.fitness.values

        # Reevaluar solo los individuos que cambiaron
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        for ind in invalid_ind:
            ind.fitness.values = toolbox.evaluate(ind)

        # Reemplazo
        pop[:] = offspring

        # Mejor individuo de la generación
        best = tools.selBest(pop, 1)[0]
        print(f"Gen {gen:3d} | Mejor fitness: {best.fitness.values[0]}")

        # Si encontramos un horario perfecto (0 violaciones), podemos parar antes
        if best.fitness.values[0] == 0:
            print("¡Horario sin violaciones encontrado, deteniendo evolución!")
            break

    # Mejor solución final
    best = tools.selBest(pop, 1)[0]
    print("\nMejor solución encontrada con violaciones:", best.fitness.values[0])
    print_schedule(best)

    return best

if __name__ == "__main__":
    main()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 4.3 MB/s eta 0:00:00
Población inicial evaluada
Gen   1 | Mejor fitness: 21.0
Gen   2 | Mejor fitness: 20.0
Gen   3 | Mejor fitness: 15.0
Gen   4 | Mejor fitness: 15.0
Gen   5 | Mejor fitness: 16.0
Gen   6 | Mejor fitness: 15.0
Gen   7 | Mejor fitness: 13.0
Gen   8 | Mejor fitness: 9.0
Gen   9 | Mejor fitness: 10.0
Gen  10 | Mejor fitness: 10.0
Gen  11 | Mejor fitness: 9.0
Gen  12 | Mejor fitness: 9.0
Gen  13 | Mejor fitness: 9.0
Gen  14 | Mejor fitness: 8.0
Gen  15 | Mejor fitness: 8.0
Gen  16 | Mejor fitness: 6.0
Gen  17 | Mejor fitness: 6.0
Gen  18 | Mejor fitness: 5.0
Gen  19 | Mejor fitness: 6.0
Gen  20 | Mejor fitness: 4.0
Gen  21 | Mejor fitness: 4.0
Gen  22 | Mejor fitness: 4.0
Gen  23 | Mejor fitness: 3.0
Gen  24 | Mejor fitness: 3.0
Gen  25 | Mejor fitness: 2.0
Gen  26 | Mejor fitness: 2.0
Gen  27 | Mejor fitness: 2.0
Gen  28 | Mejor fitness: 2.0
Gen  29 | Mejor fitness: 2.0
Gen  30 | Mejor fitness: 1.0
Gen  31 | Mejo